In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import os
pd.options.mode.chained_assignment = None
import optuna
import time
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, train_test_split
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
import pickle
from collections import defaultdict


In [34]:
depths = ["in_3_4"] #"in_1_2", "in_2_3", "in_3_4"]
for depth in depths: 
    # Cargamos los csv de los tifs
    path = "saved_files/dataset"
    dfs = {}
    for archivo in os.listdir(path):
        if archivo.endswith(f"{depth}_features.csv"):
            nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
            ruta_completa = os.path.join(path, archivo)
            dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)



    # Limpiamos valores nulos
    for nombre_df, df in dfs.items():
        for band_set in ["rhow", "rhown","rtoa"]:
            dfs[nombre_df] = df.dropna()

    dfs_to_keep = [
		'C2X-Complex_rhow_9x9_depth_in_0_1', 
		'TOA_15x15_depth_in_0_1',
		'C2X-Complex_rhown_9x9_depth_in_0_1',
		'C2X-Complex_rhow_5x5_depth_in_0_1',
		'C2RCC_rhow_5x5_depth_in_0_1',
		'C2RCC_rhow_15x15_depth_in_0_1', 
		'C2X-Complex_rhown_15x15_depth_in_0_1',
		'C2RCC_rhown_5x5_depth_in_0_1', 
		'C2X-Complex_rhow_15x15_depth_in_0_1',
		'C2RCC_rhow_9x9_depth_in_0_1',
		'C2X-Complex_rhow_5x5_depth_in_1_2', 
		'C2X_rhow_3x3_depth_in_1_2',
		'C2X-Complex_rhown_5x5_depth_in_1_2',
		'C2X-Complex_rhow_9x9_depth_in_1_2',
		'C2X-Complex_rhow_3x3_depth_in_1_2',
		'C2X-Complex_rhown_3x3_depth_in_1_2',
		'C2RCC_rhown_3x3_depth_in_1_2',
		'C2X-Complex_rhown_9x9_depth_in_1_2',
		'C2X-Complex_rhow_15x15_depth_in_1_2', 
		'C2X_rhow_5x5_depth_in_1_2',
        'TOA_15x15_depth_in_2_3',
		'TOA_9x9_depth_in_2_3',
		'TOA_5x5_depth_in_2_3', 
		'C2X-Complex_rhow_5x5_depth_in_2_3',
		'C2RCC_rhown_5x5_depth_in_2_3', 
		'TOA_3x3_depth_in_2_3',
		'C2X-Complex_rhown_5x5_depth_in_2_3', 
		'C2RCC_rhow_3x3_depth_in_2_3',
		'C2X-Complex_rhown_9x9_depth_in_2_3', 
		'C2X_rhow_9x9_depth_in_2_3',
        'TOA_9x9_depth_in_3_4',
		'TOA_3x3_depth_in_3_4',
		'TOA_5x5_depth_in_3_4',
		'C2X-Complex_rhow_5x5_depth_in_3_4', 
		'TOA_1x1_depth_in_3_4',
		'TOA_15x15_depth_in_3_4',
		'C2X-Complex_rhown_5x5_depth_in_3_4',
		'C2X-Complex_rhow_9x9_depth_in_3_4',
		'C2X-Complex_rhow_15x15_depth_in_3_4',
		'C2X-Complex_rhown_9x9_depth_in_3_4'
        ]

    dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}


    for nombre_df, df in dfs.items():
        # Marcamos las columnas de CHl alta (equivalente a quantile(0.93))
        df["High_Chl"] = df["Chl"]>5
        # Sacamos la estación de cada fecha
        df['Date'] = pd.to_datetime(df['Date'])
        def get_season(month):
            if month in [12, 1, 2]:
                return 'Invierno'
            elif month in [3, 4, 5]:
                return 'Primavera'
            elif month in [6, 7, 8]:
                return 'Verano'
            else:
                return 'Otoño'
        df['Season'] = df['Date'].dt.month.apply(get_season)
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].astype('category')
        df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
        dfs[nombre_df] = df

    #results = cross_validation_training(dfs, depth)
    # Run optuna

### Optimización de hiperparámetros

In [46]:
def objective(trial, df, nombre_df, target, model_name):
    # Mismo proceso que para validación cruzada, pero haciendo preds solamente sobre test
    FOLDS = 5
    # Para hacer clip a las predicciones y forzar > 0
    correct = True
    #results = {}

    df = df.iloc[:, 4:]

    # Separamos el conjunto de datos en train y test: Train 75% Test 25%
    train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["High_Chl"])
    target = "Chl"

    X = train.drop(columns=[target, "High_Chl", "Turbidez"])
    y = train[target]
    y_class = train["High_Chl"]

    # Definimos X e y para test
    X_test = test.drop(columns=[target, "High_Chl", "Turbidez"])
    y_test = test[target]

    #test_preds = {name: np.zeros(len(test)) for name in models}
    test_preds = np.zeros(len(test))

    #results[nombre_df] = {name: {'RMSE': [], 'R2': []} for name in models}
    # Stratified KFold de 5 folds
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

    if model_name == "LBM":
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'device': 'cpu',  # usa CPU/GPU
            'verbosity': -1,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'num_leaves': trial.suggest_categorical('num_leaves', [10, 20, 30]),
            'max_depth': trial.suggest_int('max_depth', 3, 6),
            'min_child_samples': trial.suggest_int('min_child_samples', 4, 8),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'n_estimators': trial.suggest_categorical('n_estimators', [100, 250, 500]),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'min_split_gain': trial.suggest_categorical('min_split_gain', [0.0, 0.1, 0.2, 0.5, 1.0])
        }

    if model_name == "XGB":
        params = {
            'n_estimators': trial.suggest_categorical('n_estimators', [100, 250, 00]),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 6),
            'min_child_weight': trial.suggest_int('min_child_weight', 2, 4),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'device': 'cpu',
            'objective': 'reg:squarederror',
            'tree_method': 'hist',
            'enable_categorical': True,
            'eval_metric': 'rmse'
        }
    
    if model_name == "MLP":
        # Así para evitar warning de definir como tuple
        hidden_options = {
            '32': (32,),
            '64': (64,),
            '32_16': (32, 16)
        }
        hidden_layer_sizes = trial.suggest_categorical('hidden_layer_sizes', list(hidden_options.values()))
        params = {
            'hidden_layer_sizes': hidden_layer_sizes,
            #'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', [(50,), (100,), (100, 50), (128, 64)]),
            'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
            'solver': trial.suggest_categorical('solver', ['adam', 'sgd']),
            'alpha': trial.suggest_float('alpha', 1e-4, 1, log=True),
            'learning_rate': trial.suggest_categorical('learning_rate', ['constant', 'adaptive']),
            'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'max_iter': 200,
            'n_iter_no_change': 25,
            'early_stopping': True,
            'validation_fraction': 0.2,
            'random_state': 42,
            'verbose': False
        }

    if model_name == "SVR":
        params = {
            'kernel': 'rbf',
            'C': trial.suggest_float('C', 0.1, 5.0, log=True),
            'epsilon': trial.suggest_float('epsilon', 0.01, 0.2),
            'gamma': 'scale',
            'shrinking': True,
            'tol': 1e-3,
            'max_iter': -1,
            'verbose': False
        }

    if model_name == "KNN":
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 5, 15),
            'weights': 'distance',
            'algorithm': 'auto',
            'leaf_size': trial.suggest_int('leaf_size', 10, 40),
            'p': 2,  # 1 = manhattan, 2 = euclídea
            'metric': 'minkowski',
            'n_jobs': -1
        }

    if model_name == "LR":
        # No sirve de mucho, pero por completitud
        params = {
            'fit_intercept': trial.suggest_categorical('fit_intercept', [True, False]),
            'positive': trial.suggest_categorical('positive', [True, False]),
        }

    if model_name == "RF":
        params = {
            'n_estimators': trial.suggest_categorical('n_estimators', [100, 300, 500]),
            'max_depth': trial.suggest_int('max_depth', 3, 6),
            'min_samples_split': trial.suggest_int('min_samples_split', 4, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 8),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
            'random_state': 42,
            'verbose': 0
        }

    if model_name == "CAT":
        params = {
            'iterations': trial.suggest_categorical('iterations', [500, 750, 100]),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),
            'depth': trial.suggest_int('depth', 4, 8),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 2.0, 1.0),
            'loss_function': 'RMSE',
            'eval_metric': 'RMSE',
            'random_seed': 42,
            'early_stopping_rounds': 50,
            'verbose': False,
            'random_strength': trial.suggest_float('random_strength', 0.0, 1.0),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        }

    if model_name == "ELN":
        params = {
            'alpha': trial.suggest_float('alpha', 1e-4, 1.0, log=True),
            'l1_ratio': trial.suggest_float('l1_ratio', 0.1, 0.9),
            'fit_intercept': True,
            'max_iter': 1000,
            'tol': 1e-4,
            'selection': 'cyclic',
            'random_state': 42
        }

    # Cargamos el modelo correspondiente
    model = models[model_name](**params)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        print(f"Fold {fold+1}")
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if model_name in ["MLP", "SVR", "KNN", "LR", "ELN"]:
            scaler_X = RobustScaler()
            scaler_y = RobustScaler()
            X_train_scaled = scaler_X.fit_transform(X_train)
            X_val_scaled = scaler_X.transform(X_val)
            X_test_scaled = scaler_X.transform(X_test)
            y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
            model.fit(X_train_scaled, y_train_scaled)
            test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()

        else:
            model.fit(X_train, y_train)
            test_pred = model.predict(X_test)

        if correct:
            test_pred = np.clip(test_pred, 0.2, None)

        test_preds[model_name] += test_pred / FOLDS

    #rmse_test = np.sqrt(mean_squared_error(y_test, test_preds[model_name]))
    r2_test = r2_score(y_test, test_preds[model_name])

    return r2_test

In [36]:

models = {
    "XGB": XGBRegressor,
    "LBM": LGBMRegressor,
    "MLP": MLPRegressor,
    "SVR": SVR,
    "KNN": KNeighborsRegressor,
    "LR": LinearRegression,
    "RF": RandomForestRegressor,
    "CAT": CatBoostRegressor,
    "ELN":  ElasticNet
}

def run_optuna(df, nombre_df, target, n_trials, model_name):
    results = {}

    print(f"Buscando mejores hiperparámetros para {model_name} con {nombre_df}...")
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, df, nombre_df, target, model_name), n_trials=n_trials, timeout= 1500)
    
    print(f"\n✅ {model_name} con {nombre_df} - Mejor R2: {study.best_value:.2f}")
    print(f"📋 Parámetros: {study.best_params}\n")
    
    results[model_name] = {
        'best_params': study.best_params,
        'best_score': np.round(study.best_value, 3),
        'study': study
    }
    return results

n_trials = 20
global_results = {}


depths = ["in_0_1", "in_1_2", "in_2_3", "in_3_4"]
for depth in depths: 
    # Cargamos los csv de los tifs
    path = "saved_files/dataset"
    dfs = {}
    for archivo in os.listdir(path):
        if archivo.endswith(f"{depth}_features.csv"):
            nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
            ruta_completa = os.path.join(path, archivo)
            dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)


    # Limpiamos valores nulos
    for nombre_df, df in dfs.items():
        for band_set in ["rhow", "rhown","rtoa"]:
            dfs[nombre_df] = df.dropna()

    dfs_to_keep = [
		'C2X-Complex_rhow_9x9_depth_in_0_1', 
		'TOA_15x15_depth_in_0_1',
		'C2X-Complex_rhown_9x9_depth_in_0_1',
		'C2X-Complex_rhow_5x5_depth_in_0_1',
		'C2RCC_rhow_5x5_depth_in_0_1',
		'C2RCC_rhow_15x15_depth_in_0_1', 
		'C2X-Complex_rhown_15x15_depth_in_0_1',
		'C2RCC_rhown_5x5_depth_in_0_1', 
		'C2X-Complex_rhow_15x15_depth_in_0_1',
		'C2RCC_rhow_9x9_depth_in_0_1',
		'C2X-Complex_rhow_5x5_depth_in_1_2', 
		'C2X_rhow_3x3_depth_in_1_2',
		'C2X-Complex_rhown_5x5_depth_in_1_2',
		'C2X-Complex_rhow_9x9_depth_in_1_2',
		'C2X-Complex_rhow_3x3_depth_in_1_2',
		'C2X-Complex_rhown_3x3_depth_in_1_2',
		'C2RCC_rhown_3x3_depth_in_1_2',
		'C2X-Complex_rhown_9x9_depth_in_1_2',
		'C2X-Complex_rhow_15x15_depth_in_1_2', 
		'C2X_rhow_5x5_depth_in_1_2',
        'TOA_15x15_depth_in_2_3',
		'TOA_9x9_depth_in_2_3',
		'TOA_5x5_depth_in_2_3', 
		'C2X-Complex_rhow_5x5_depth_in_2_3',
		'C2RCC_rhown_5x5_depth_in_2_3', 
		'TOA_3x3_depth_in_2_3',
		'C2X-Complex_rhown_5x5_depth_in_2_3', 
		'C2RCC_rhow_3x3_depth_in_2_3',
		'C2X-Complex_rhown_9x9_depth_in_2_3', 
		'C2X_rhow_9x9_depth_in_2_3',
        'TOA_9x9_depth_in_3_4',
		'TOA_3x3_depth_in_3_4',
		'TOA_5x5_depth_in_3_4',
		'C2X-Complex_rhow_5x5_depth_in_3_4', 
		'TOA_1x1_depth_in_3_4',
		'TOA_15x15_depth_in_3_4',
		'C2X-Complex_rhown_5x5_depth_in_3_4',
		'C2X-Complex_rhow_9x9_depth_in_3_4',
		'C2X-Complex_rhow_15x15_depth_in_3_4',
		'C2X-Complex_rhown_9x9_depth_in_3_4'
        ]

    dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}


    for nombre_df, df in dfs.items():
        # Marcamos las columnas de CHl alta (equivalente a quantile(0.93))
        df["High_Chl"] = df["Chl"]>5
        # Sacamos la estación de cada fecha
        df['Date'] = pd.to_datetime(df['Date'])
        def get_season(month):
            if month in [12, 1, 2]:
                return 'Invierno'
            elif month in [3, 4, 5]:
                return 'Primavera'
            elif month in [6, 7, 8]:
                return 'Verano'
            else:
                return 'Otoño'
        df['Season'] = df['Date'].dt.month.apply(get_season)
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].astype('category')
        df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
        dfs[nombre_df] = df

    for nombre_df, df in list(dfs.items()):
        for model_name in models.keys():
            key = (nombre_df, model_name)
            result = run_optuna(df, nombre_df, "Chl", n_trials, model_name)
            global_results[key] = result[model_name]

    with open(f"training_results/selection_results_{depth}.pkl", "wb") as f:
        pickle.dump(global_results, f)

[I 2025-09-04 14:19:12,167] A new study created in memory with name: no-name-3ec5fe63-a087-4177-87eb-4b637483cb3c


Buscando mejores hiperparámetros para XGB con TOA_9x9_depth_in_3_4...

=== Procesando TOA_9x9_depth_in_3_4 ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-09-04 14:19:15,176] Trial 0 finished with value: 0.5200771062651086 and parameters: {'n_estimators': 500, 'learning_rate': 0.010154558822693555, 'max_depth': 5, 'min_child_weight': 1, 'subsample': 0.8202811812149784, 'colsample_bytree': 0.6729905646697188}. Best is trial 0 with value: 0.5200771062651086.



=== Procesando TOA_9x9_depth_in_3_4 ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-09-04 14:19:23,159] Trial 1 finished with value: 0.47749786193414134 and parameters: {'n_estimators': 1000, 'learning_rate': 0.014900629764633438, 'max_depth': 6, 'min_child_weight': 1, 'subsample': 0.9926663347183362, 'colsample_bytree': 0.7326210265974088}. Best is trial 0 with value: 0.5200771062651086.


...

Fold 4
Fold 5

=== Procesando C2X-Complex_rhown_9x9_depth_in_3_4 ===
Fold 1
Fold 2
Fold 3
Fold 4


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.012e+01, tolerance: 4.433e-02
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.033e+01, tolerance: 5.000e-02
  model = cd_fast.enet_coordinate_descent(
[I 2025-09-04 18:40:39,530] Trial 22 finished with value: 0.5921951567857184 and parameters: {'alpha': 0.0006519825496700685, 'l1_ratio': 0.4259782426330299}. Best is trial 22 with value: 0.5921951567857184.
/home/antonio/.pye

Fold 5

=== Procesando C2X-Complex_rhown_9x9_depth_in_3_4 ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.359e+01, tolerance: 5.000e-02
  model = cd_fast.enet_coordinate_descent(
[I 2025-09-04 18:40:39,717] Trial 23 finished with value: 0.5794362072702861 and parameters: {'alpha': 0.002314283053760167, 'l1_ratio': 0.4314628852963102}. Best is trial 22 with value: 0.5921951567857184.
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.189e-01, tolerance: 4.433e-02
  model = cd_fast.enet_coordinate_descent(
[I 2025-09-04 18:40


=== Procesando C2X-Complex_rhown_9x9_depth_in_3_4 ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

✅ ELN con C2X-Complex_rhown_9x9_depth_in_3_4 - Mejor R2: 0.59
📋 Parámetros: {'alpha': 0.0006519825496700685, 'l1_ratio': 0.4259782426330299}



In [292]:
def objective(trial, df, target, model_name):
    # df = df.iloc[:,4:]
    # # Mismos splits que para el entrenamiento
    # train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["High_Chl"]) # TEST 20% TRAIN 80%
    # target = "Chl"
    # train, val = train_test_split(train, test_size=0.25, random_state=42, stratify=train["High_Chl"])

    # X = train.drop(columns=[target, 'High_Chl'])
    # y = train[target]
    # y_class = train["High_Chl"]

    # Definimos los folds
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_preds = np.zeros(len(train))  # Almacenar las predicciones OOF

    start = time.time()
    
    if model_name == "LBM":
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'device': 'cpu',  # usa CPU/GPU
            'verbosity': -1,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_categorical('num_leaves', [20, 40, 60, 80]),
            'max_depth': trial.suggest_int('max_depth', 5, 8),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 25),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'n_estimators': trial.suggest_categorical('n_estimators', [500, 1000, 2000])
        }

    if model_name == "XGB":
        params = {
            'n_estimators': trial.suggest_categorical('n_estimators', [500, 1000, 2000]),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 5, 8),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 4),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'device': 'cpu',
            'objective': 'reg:squarederror',
            'tree_method': 'hist',
            'enable_categorical': True,
            'eval_metric': 'rmse'
        }
    
    if model_name == "MLP":
        # Así para evitar warning de definir como tuple
        hidden_options = {
            '50': (50,),
            '100': (100,),
            '100_50': (100, 50),
            '128_64': (128, 64)
        }
        key = trial.suggest_categorical('hidden_layer_sizes', list(hidden_options.keys()))
        params = {
            'hidden_layer_sizes': hidden_options[key],
            #'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', [(50,), (100,), (100, 50), (128, 64)]),
            'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
            'solver': trial.suggest_categorical('solver', ['adam', 'sgd']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
            'learning_rate': trial.suggest_categorical('learning_rate', ['constant', 'adaptive']),
            'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'max_iter': 200,
            'n_iter_no_change': 25,
            'early_stopping': True,
            'validation_fraction': 0.2,
            'random_state': 42,
            'verbose': False
        }

    if model_name == "SVR":
        params = {
            'kernel': trial.suggest_categorical('kernel', ['rbf', 'sigmoid']),
            'C': trial.suggest_float('C', 0.1, 10.0, log=True),
            'epsilon': trial.suggest_float('epsilon', 0.01, 0.2),
            'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
            'shrinking': True,
            'tol': 1e-3,
            'max_iter': -1,
            'verbose': False
        }

    if model_name == "KNN":
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 3, 15),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'algorithm': 'auto',
            'leaf_size': trial.suggest_int('leaf_size', 10, 40),
            'p': 2,  # 1 = manhattan, 2 = euclídea
            'metric': 'minkowski',
            'n_jobs': -1
        }

    if model_name == "LR":
        # No sirve de mucho, pero por completitud
        params = {
            'fit_intercept': trial.suggest_categorical('fit_intercept', [True, False]),
            'positive': trial.suggest_categorical('positive', [True, False]),
        }

    if model_name == "RF":
        params = {
            'n_estimators': trial.suggest_categorical('n_estimators', [100, 300, 500]),
            'max_depth': trial.suggest_int('max_depth', 5, 15),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
            'random_state': 42,
            'verbose': 0
        }

    if model_name == "CAT":
        params = {
            'iterations': trial.suggest_categorical('iterations', [500, 1000, 2000]),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
            'loss_function': 'RMSE',
            'eval_metric': 'RMSE',
            'random_seed': 42,
            'early_stopping_rounds': 50,
            'verbose': False,
        }

    if model_name == "EN":
        params = {
            'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True),
            'l1_ratio': trial.suggest_float('l1_ratio', 0.0, 1.0),
            'fit_intercept': True,
            'max_iter': 1000,
            'tol': 1e-4,
            'selection': 'cyclic',
            'random_state': 42
        }

    # Cargamos el modelo correspondiente
    model = models[model_name](**params)

    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        print(f"Fold {fold+1}")
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if model_name in ["MLP", "SVR", "KNN", "LR", "EN"]:
            scaler_X = RobustScaler()
            scaler_y = RobustScaler()
            X_train_scaled = scaler_X.fit_transform(X_train)
            X_val_scaled = scaler_X.transform(X_val)
            y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
            model.fit(X_train_scaled, y_train_scaled)
            y_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
        
        oof_preds[val_idx] = y_pred


    print(f"Running time: {time.time() - start:.1f} sec")
    # Calculamos el RMSE OOF
    rmse_score = np.sqrt(mean_squared_error(y, oof_preds))
    r2 = r2_score(y, oof_preds)
    print(f"OOF RMSE: {rmse_score:.2f} | R2: {r2:.2f}")
    
    return r2

In [47]:
global_results

{('TOA_9x9_depth_in_3_4',
  'XGB'): {'best_params': {'n_estimators': 2000,
   'learning_rate': 0.047600594240155655,
   'max_depth': 5,
   'min_child_weight': 4,
   'subsample': 0.6049475476360661,
   'colsample_bytree': 0.8631912096943319}, 'best_score': 0.615, 'study': <optuna.study.study.Study at 0x7800b0502610>},
 ('TOA_9x9_depth_in_3_4',
  'LBM'): {'best_params': {'learning_rate': 0.01713416804277511,
   'num_leaves': 80,
   'max_depth': 8,
   'min_child_samples': 11,
   'subsample': 0.7883599431510695,
   'colsample_bytree': 0.8460115625974576,
   'n_estimators': 500}, 'best_score': 0.628, 'study': <optuna.study.study.Study at 0x7800b0502c10>},
 ('TOA_9x9_depth_in_3_4',
  'MLP'): {'best_params': {'hidden_layer_sizes': '256_128',
   'activation': 'relu',
   'solver': 'sgd',
   'alpha': 0.015492411781384838,
   'learning_rate': 'adaptive',
   'learning_rate_init': 0.0019024503727906594}, 'best_score': 0.71, 'study': <optuna.study.study.Study at 0x7800b014d100>},
 ('TOA_9x9_depth_in

In [14]:
with open(f"training_results/selection_results_{depth}.pkl", "wb") as f:
    pickle.dump(global_results, f)

In [39]:
# Diccionario para agrupar parámetros por modelo
params_by_model = defaultdict(list)

# Agrupar best_params por modelo
for (df_name, model_name), result in global_results.items():
    best_params = result["best_params"]
    params_by_model[model_name].append(best_params)

# Crear DataFrames con medias y std por modelo
summary_stats = {}

for model_name, param_list in params_by_model.items():
    df_params = pd.DataFrame(param_list)

    # Filtramos solo columnas numéricas para calcular medias y std
    df_numeric = df_params.select_dtypes(include=[np.number])

    stats = pd.concat([df_numeric.mean().rename("mean"), df_numeric.std().rename("std")], axis=1)
    summary_stats[model_name] = stats

# Mostrar un ejemplo
summary_stats["LBM"]

,mean,std
learning_rate,0.011264,0.006322
num_leaves,52.000000,21.499354
max_depth,6.800000,1.032796
min_child_samples,9.000000,3.366502
subsample,0.846105,0.135599
colsample_bytree,0.853437,0.094603
n_estimators,550.000000,158.113883


In [37]:
global_results

{('TOA_9x9_depth_in_3_4',
  'XGB'): {'best_params': {'n_estimators': 2000,
   'learning_rate': 0.047600594240155655,
   'max_depth': 5,
   'min_child_weight': 4,
   'subsample': 0.6049475476360661,
   'colsample_bytree': 0.8631912096943319}, 'best_score': 0.615, 'study': <optuna.study.study.Study at 0x7800b0502610>},
 ('TOA_9x9_depth_in_3_4',
  'LBM'): {'best_params': {'learning_rate': 0.01713416804277511,
   'num_leaves': 80,
   'max_depth': 8,
   'min_child_samples': 11,
   'subsample': 0.7883599431510695,
   'colsample_bytree': 0.8460115625974576,
   'n_estimators': 500}, 'best_score': 0.628, 'study': <optuna.study.study.Study at 0x7800b0502c10>},
 ('TOA_9x9_depth_in_3_4',
  'MLP'): {'best_params': {'hidden_layer_sizes': '256_128',
   'activation': 'relu',
   'solver': 'sgd',
   'alpha': 0.015492411781384838,
   'learning_rate': 'adaptive',
   'learning_rate_init': 0.0019024503727906594}, 'best_score': 0.71, 'study': <optuna.study.study.Study at 0x7800b014d100>},
 ('TOA_9x9_depth_in

**Entrenamiento con los parámetros seleccionados**

In [318]:
results = {}

for nombre_df, df in list(dfs.items()):
    #print(nombre_df)
    df = df.iloc[:,4:]

    # Para usar solamente bandas, sin combinaciones
    # if 'TOA' in nombre_df:
    #     # TOA solamente con las bandas, parece que las combinaciones solo meten ruido
    #     df = df.iloc[:,np.r_[0:14, 58:60]]
    # if 'rhow' in nombre_df and 'rhown' not in nombre_df:
    #     df = df.iloc[:,np.r_[0:9, 53:55]]
    # if 'rhown' in nombre_df:
    #     df = df.iloc[:,np.r_[0:7, 51:53]]

    train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["High_Chl"]) # TEST 20% TRAIN 80%
    target = "Chl"

    train, val = train_test_split(train, test_size=0.25, random_state=42, stratify=train["High_Chl"]) # TRAIN 60% VAL 20% TEST%

    X_train = train.drop(columns=[target,"High_Chl"])
    X_val = val.drop(columns=[target, "High_Chl"])
    X_test = test.drop(columns=[target, "High_Chl"])
    y_train = train[target]
    y_val = val[target]
    y_test = test[target]

    
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_val_scaled = scaler_X.transform(X_val)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
    
    results[nombre_df] = {name: {'RMSE': None, 'R2': None} for name in models}
    val_preds = {}
    test_preds = {}

    for name, model in models.items():
        print(f"Fitting {name} for {nombre_df}")
        if name in ["MLP", "SVR", "KNN", "LR", "EN"]:
            model.fit(X_train_scaled, y_train_scaled)
            #test_pred = model.predict(X_test_scaled)
            val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
            test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
        else:
            model.fit(X_train, y_train)
            val_pred = model.predict(X_val)
            test_pred = model.predict(X_test)

        val_preds[name] = val_pred
        test_preds[name] = test_pred

        rmse = np.sqrt(mean_squared_error(y_test, test_pred))
        r2 = r2_score(y_test, test_pred)

        results[nombre_df][name]['RMSE'] = rmse.round(2)
        results[nombre_df][name]['R2'] = r2.round(2)

    # Meta-modelo
    meta_X = np.vstack([val_preds[model] for model in models]).T
    meta_y = y_val.values
    meta_model = Ridge().fit(meta_X, meta_y)

    # Predicción final ensemble
    test_meta_X = np.vstack([test_preds[model] for model in models]).T
    ensemble_pred = meta_model.predict(test_meta_X)

    rmse_ens = np.sqrt(mean_squared_error(y_test, ensemble_pred))
    r2_ens = r2_score(y_test, ensemble_pred)

    results[nombre_df]["Ensemble"] = {
        "RMSE": round(rmse_ens, 2),
        "R2": round(r2_ens, 2)
    }

Fitting XGB for C2RCC_rhow_5x5_depth_lt_1
Fitting LBM for C2RCC_rhow_5x5_depth_lt_1
Fitting MLP for C2RCC_rhow_5x5_depth_lt_1
Fitting SVR for C2RCC_rhow_5x5_depth_lt_1
Fitting KNN for C2RCC_rhow_5x5_depth_lt_1
Fitting LR for C2RCC_rhow_5x5_depth_lt_1
Fitting RF for C2RCC_rhow_5x5_depth_lt_1
Fitting CAT for C2RCC_rhow_5x5_depth_lt_1
Fitting EN for C2RCC_rhow_5x5_depth_lt_1
Fitting XGB for C2X-Complex_rhown_3x3_depth_lt_1
Fitting LBM for C2X-Complex_rhown_3x3_depth_lt_1
Fitting MLP for C2X-Complex_rhown_3x3_depth_lt_1
Fitting SVR for C2X-Complex_rhown_3x3_depth_lt_1
Fitting KNN for C2X-Complex_rhown_3x3_depth_lt_1
Fitting LR for C2X-Complex_rhown_3x3_depth_lt_1
Fitting RF for C2X-Complex_rhown_3x3_depth_lt_1
Fitting CAT for C2X-Complex_rhown_3x3_depth_lt_1
Fitting EN for C2X-Complex_rhown_3x3_depth_lt_1
Fitting XGB for TOA_9x9_depth_lt_1
Fitting LBM for TOA_9x9_depth_lt_1
Fitting MLP for TOA_9x9_depth_lt_1
Fitting SVR for TOA_9x9_depth_lt_1
Fitting KNN for TOA_9x9_depth_lt_1
Fitting LR f

In [329]:
rows = []

for df_name, model_scores in results.items():
    row = {}
    for model_name, metrics in model_scores.items():
        for metric_name, values in metrics.items():
            if isinstance(values, list):  # Solo para los que tienen listas (folds)
                mean_val = np.mean(values)
                std_val = np.std(values)
                row[(metric_name, model_name)] = f"{mean_val:.2f} ± {std_val:.2f}"
            else:
                # Para el ensemble que tiene un único valor
                row[(metric_name, model_name)] = f"{values:.2f}"
    rows.append((df_name, row))

df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
df_results = df_results.sort_index(axis=1, level=0)
df_results = df_results.sort_index(axis=0)


In [330]:
df_results

Metric                                     R2                         \
Model                                     CAT           EN  Ensemble   
C2RCC_rhow_5x5_depth_lt_1         0.67 ± 0.13  0.48 ± 0.07      0.61   
C2X-Complex_rhown_3x3_depth_lt_1  0.66 ± 0.12  0.50 ± 0.06  -6049.03   
TOA_9x9_depth_lt_1                0.78 ± 0.12  0.19 ± 0.13      0.46   

Metric                                                                    \
Model                                     KNN          LBM            LR   
C2RCC_rhow_5x5_depth_lt_1         0.74 ± 0.10  0.61 ± 0.24   0.31 ± 0.52   
C2X-Complex_rhown_3x3_depth_lt_1  0.59 ± 0.12  0.55 ± 0.17  -1.12 ± 3.18   
TOA_9x9_depth_lt_1                0.76 ± 0.13  0.70 ± 0.09   0.18 ± 0.36   

Metric                                                                   \
Model                                     MLP           RF          SVR   
C2RCC_rhow_5x5_depth_lt_1         0.71 ± 0.12  0.63 ± 0.18  0.66 ± 0.15   
C2X-Complex_rhown_3x3_depth_lt_1  0.45 ± 0.08  0.63 ± 0.11  0.53 ± 0.09   
TOA_9x9_depth_lt_1                0.65 ± 0.14  0.70 ± 0.13  0.50 ± 0.04   

Metric                                                RMSE               \
Model                                     XGB          CAT           EN   
C2RCC_rhow_5x5_depth_lt_1         0.65 ± 0.19  2.04 ± 0.37  2.64 ± 0.20   
C2X-Complex_rhown_3x3_depth_lt_1  0.61 ± 0.17  2.08 ± 0.17  2.63 ± 0.41   
TOA_9x9_depth_lt_1                0.72 ± 0.12  1.80 ± 0.49  3.58 ± 0.15   

Metric                                                               \
Model                            Ensemble          KNN          LBM   
C2RCC_rhow_5x5_depth_lt_1            2.97  1.81 ± 0.21  2.16 ± 0.50   
C2X-Complex_rhown_3x3_depth_lt_1   370.82  2.30 ± 0.17  2.41 ± 0.25   
TOA_9x9_depth_lt_1                   2.59  1.87 ± 0.37  2.15 ± 0.32   

Metric                                                                   \
Model                                      LR          MLP           RF   
C2RCC_rhow_5x5_depth_lt_1         2.83 ± 0.72  1.93 ± 0.27  2.14 ± 0.35   
C2X-Complex_rhown_3x3_depth_lt_1  4.08 ± 2.64  2.77 ± 0.44  2.21 ± 0.17   
TOA_9x9_depth_lt_1                3.49 ± 0.48  2.31 ± 0.28  2.12 ± 0.39   

Metric                                                      
Model                                     SVR          XGB  
C2RCC_rhow_5x5_depth_lt_1         2.08 ± 0.31  2.08 ± 0.44  
C2X-Complex_rhown_3x3_depth_lt_1  2.52 ± 0.31  2.21 ± 0.28  
TOA_9x9_depth_lt_1                2.84 ± 0.26  2.05 ± 0.44